In [10]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import spacy
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
import gensim.downloader as api
from sklearn.metrics.pairwise import cosine_similarity
import torch
from transformers import AutoTokenizer, AutoModel
from transformers import pipeline

In [11]:
df = pd.read_csv("../data/clean_hr_data.csv")

In [12]:
measure_rank_1 = "aspiring human resources" 
measure_rank_2 =  "seeking human resources"

In [13]:
# BOW method 
# You fit and transform on the column
# you only transform the y1 and y2
vectorizer_bow = CountVectorizer()
X = vectorizer_bow.fit_transform(df['clean_job_title'])
y1 = vectorizer_bow.transform([measure_rank_1])
y2 = vectorizer_bow.transform([measure_rank_2])

score_1 = cosine_similarity(y1, X)
score_2 = cosine_similarity(y2, X)

df["similarity_1_bow"] = score_1.flatten()
df[df["similarity_1_bow"]>0.3][["clean_job_title", "similarity_1_bow"]].sort_values(
    by="similarity_1_bow",
    ascending=False
).head(10)

,clean_job_title,similarity_1_bow
21,aspiring human resources manager seek internsh...,0.745356
47,aspiring human resources manager | graduating ...,0.615457
5,aspiring human resources specialist greater ne...,0.577350
6,student humber college aspiring human resource...,0.577350
20,business management major aspiring human resou...,0.547723
2,aspiring human resources professional raleigh ...,0.547723
12,aspiring human resources management student se...,0.522233
30,aspiring human resources professional | energe...,0.500000
13,seek human resources opportunities chicago ill...,0.436436
36,human resources management major milpitas cali...,0.436436


In [14]:
df["similarity_2_bow"] = score_2.flatten()
df[df["similarity_2_bow"]>0.3][["clean_job_title", "similarity_2_bow"]].sort_values(
    by="similarity_2_bow",
    ascending=False
).head(10)

,clean_job_title,similarity_2_bow
21,aspiring human resources manager seek internsh...,0.730297
47,aspiring human resources manager | graduating ...,0.603023
13,seek human resources opportunities chicago ill...,0.534522
22,human resources professional greater boston ar...,0.534522
36,human resources management major milpitas cali...,0.534522
37,director human resources ey greater atlanta ...,0.500000
26,human resources generalist schwan amerika birl...,0.500000
15,human resources staffing recruiting profession...,0.471405
46,seek human resources position las vegas nevada...,0.471405
6,student humber college aspiring human resource...,0.471405


In [15]:
# Tfidf method
vectorizer_tfidf = TfidfVectorizer()
X = vectorizer_tfidf.fit_transform(df['clean_job_title'])
y1 = vectorizer_tfidf.transform([measure_rank_1])
y2 = vectorizer_tfidf.transform([measure_rank_2])

score_1 = cosine_similarity(y1, X)
score_2 = cosine_similarity(y2, X)


df["similarity_1_tfidf"] = score_1.flatten()
df[df["similarity_1_tfidf"]>0.3][["clean_job_title", "similarity_1_tfidf"]].sort_values(
    by="similarity_1_tfidf",
    ascending=False
).head(10)

,clean_job_title,similarity_1_tfidf
21,aspiring human resources manager seek internsh...,0.547299
5,aspiring human resources specialist greater ne...,0.416505
12,aspiring human resources management student se...,0.390410
2,aspiring human resources professional raleigh ...,0.365241
20,business management major aspiring human resou...,0.361899
6,student humber college aspiring human resource...,0.345649
47,aspiring human resources manager | graduating ...,0.308499


In [16]:
df["similarity_2_tfidf"] = score_2.flatten()
df[df["similarity_2_tfidf"]>0.3][["clean_job_title", "similarity_2_tfidf"]].sort_values(
    by="similarity_2_tfidf",
    ascending=False
).head(10)

,clean_job_title,similarity_2_tfidf
21,aspiring human resources manager seek internsh...,0.491453


In [17]:
#%pip install gensim

model = api.load('glove-wiki-gigaword-100')


# apply has the tendency to work on the rows automatically 
def get_title_vector(row):
    words = row.split()

    vectors = []

    for word in words:
        if word in model:
            vectors.append(model[word])

    return np.mean(vectors, axis=0)


    
df['vector_for_modeling'] = df["clean_job_title"].apply(get_title_vector)
y1_vector = get_title_vector(measure_rank_1)
y2_vector = get_title_vector(measure_rank_2)


X_glove = np.vstack(df["vector_for_modeling"])
# to make it work you need to use vstack function
score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_glove
)

score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    X_glove
)

df["similarity_1_glove"] = score_1.flatten()
# turns a multi-dimensional NumPy array into a one-dimensional array
df[df["similarity_1_glove"]>0.3][["clean_job_title", "similarity_1_glove"]].sort_values(
    by="similarity_1_glove",
    ascending=False
).head(10)

,clean_job_title,similarity_1_glove
21,aspiring human resources manager seek internsh...,0.849718
24,aspiring human resources professional | passio...,0.821569
13,seek human resources opportunities chicago ill...,0.811086
32,human resources professional world leader gis ...,0.809051
37,director human resources ey greater atlanta ...,0.802835
12,aspiring human resources management student se...,0.800376
8,seek human resources hris generalist positions...,0.798086
22,human resources professional greater boston ar...,0.797593
5,aspiring human resources specialist greater ne...,0.788184
30,aspiring human resources professional | energe...,0.784895


In [18]:
df["similarity_2_glove"] = score_2.flatten()
# turns a multi-dimensional NumPy array into a one-dimensional array
df[df["similarity_2_glove"]>0.3][["clean_job_title", "similarity_2_glove"]].sort_values(
    by="similarity_2_glove",
    ascending=False
).head(10)


,clean_job_title,similarity_2_glove
13,seek human resources opportunities chicago ill...,0.874183
21,aspiring human resources manager seek internsh...,0.871159
8,seek human resources hris generalist positions...,0.863006
42,seek human resources opportunity open travel...,0.862108
37,director human resources ey greater atlanta ...,0.826413
36,human resources management major milpitas cali...,0.825759
22,human resources professional greater boston ar...,0.820881
32,human resources professional world leader gis ...,0.809376
12,aspiring human resources management student se...,0.808565
24,aspiring human resources professional | passio...,0.801996


In [19]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased")

def get_title_vector_bert(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )
# pt is py_torch, truncation for fitting what the model allow in size, padding makes the size the same
    with torch.no_grad():
        outputs = bert(**inputs)


    vector = outputs.last_hidden_state.mean(dim=1)

    return vector.squeeze().numpy()
# make the tensor a numpy array with one dimension 
df['bert_for_modeling'] = df["clean_job_title"].apply(get_title_vector_bert)
y1_vector = get_title_vector(measure_rank_1)
y2_vector = get_title_vector(measure_rank_2)



X_bert = np.vstack(df["bert_for_modeling"])

y1_vector = get_title_vector_bert(measure_rank_1)
y2_vector = get_title_vector_bert(measure_rank_2)

score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_bert
)

score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    X_bert
)

df["similarity_1_bert"] = score_1.flatten()

df[df["similarity_1_bert"]>0.3][["clean_job_title", "similarity_1_bert"]].sort_values(
    by="similarity_1_bert",
    ascending=False
).head(10)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,clean_job_title,similarity_1_bert
13,seek human resources opportunities chicago ill...,0.749111
21,aspiring human resources manager seek internsh...,0.747322
30,aspiring human resources professional | energe...,0.735730
15,human resources staffing recruiting profession...,0.720959
12,aspiring human resources management student se...,0.720783
14,experienced retail manager aspire human resour...,0.719675
6,student humber college aspiring human resource...,0.719633
32,human resources professional world leader gis ...,0.715453
51,director administration excellence logging kat...,0.707068
5,aspiring human resources specialist greater ne...,0.702381


In [20]:
df["similarity_2_bert"] = score_2.flatten()

df[df["similarity_2_bert"]>0.3][["clean_job_title", "similarity_2_bert"]].sort_values(
    by="similarity_2_bert",
    ascending=False
).head(10)

,clean_job_title,similarity_2_bert
13,seek human resources opportunities chicago ill...,0.801287
21,aspiring human resources manager seek internsh...,0.702566
15,human resources staffing recruiting profession...,0.700950
8,seek human resources hris generalist positions...,0.698048
46,seek human resources position las vegas nevada...,0.684404
40,seek employment opportunity customer service p...,0.683962
51,director administration excellence logging kat...,0.683022
3,people development coordinator ryan denton tex...,0.679889
22,human resources professional greater boston ar...,0.679089
37,director human resources ey greater atlanta ...,0.666376


In [21]:
tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)

sbert = AutoModel.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)


def get_title_vector_sbert(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = sbert(**inputs)

    # Token embeddings
    token_embeddings = outputs.last_hidden_state

    # Attention mask
    attention_mask = inputs["attention_mask"]

    # Expand mask to same dimensions as embeddings
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

    # Mean pooling, ignoring padding tokens
    vector = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

    return vector.squeeze().numpy()


df["sbert_for_modeling"] = df["clean_job_title"].apply(
    get_title_vector_sbert
)


s_bert = np.vstack(df["sbert_for_modeling"])

y1_vector = get_title_vector_sbert(measure_rank_1)
y2_vector = get_title_vector_sbert(measure_rank_2)

score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    s_bert
)

score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    s_bert
)

df["similarity_1_sbert"] = score_1.flatten()

df[df["similarity_1_sbert"]>0.3][["clean_job_title", "similarity_1_sbert"]].sort_values(
    by="similarity_1_sbert",
    ascending=False
).head(10)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,clean_job_title,similarity_1_sbert
2,aspiring human resources professional raleigh ...,0.819170
5,aspiring human resources specialist greater ne...,0.788344
47,aspiring human resources manager | graduating ...,0.775312
21,aspiring human resources manager seek internsh...,0.731115
46,seek human resources position las vegas nevada...,0.710243
30,aspiring human resources professional | energe...,0.704329
6,student humber college aspiring human resource...,0.702629
12,aspiring human resources management student se...,0.701894
24,aspiring human resources professional | passio...,0.699898
13,seek human resources opportunities chicago ill...,0.670924


In [22]:
df["similarity_2_sbert"] = score_2.flatten()

df[df["similarity_2_sbert"]>0.3][["clean_job_title", "similarity_2_sbert"]].sort_values(
    by="similarity_2_sbert",
    ascending=False
).head(10)

,clean_job_title,similarity_2_sbert
46,seek human resources position las vegas nevada...,0.761732
2,aspiring human resources professional raleigh ...,0.734525
5,aspiring human resources specialist greater ne...,0.717399
13,seek human resources opportunities chicago ill...,0.717316
8,seek human resources hris generalist positions...,0.699338
48,human resources generalist loparex raleigh dur...,0.671056
47,aspiring human resources manager | graduating ...,0.661074
21,aspiring human resources manager seek internsh...,0.658575
30,aspiring human resources professional | energe...,0.653204
24,aspiring human resources professional | passio...,0.640741


In [23]:
tokenizer = AutoTokenizer.from_pretrained(
    "intfloat/e5-small-v2"
)

e5 = AutoModel.from_pretrained(
    "intfloat/e5-small-v2"
)


def get_title_vector_e5(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = e5(**inputs)

    # Token embeddings
    token_embeddings = outputs.last_hidden_state

    # Attention mask
    attention_mask = inputs["attention_mask"]

    # Expand mask to same dimensions as embeddings
    mask = attention_mask.unsqueeze(-1).expand(
        token_embeddings.size()
    ).float()

    # Mean pooling
    vector = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

    return vector.squeeze().numpy()


df["e5_for_modeling"] = df["clean_job_title"].apply(
    get_title_vector_e5
)




X_e5 = np.vstack(df["e5_for_modeling"])

y1_vector = get_title_vector_e5(measure_rank_1)
y2_vector = get_title_vector_e5(measure_rank_2)


score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_e5
)

df["similarity_1_5e"] = score_1.flatten()

df[df["similarity_1_5e"]>0.3][["clean_job_title", "similarity_1_5e"]].sort_values(
    by="similarity_1_5e",
    ascending=False
).head(10)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,clean_job_title,similarity_1_5e
5,aspiring human resources specialist greater ne...,0.900310
21,aspiring human resources manager seek internsh...,0.894750
47,aspiring human resources manager | graduating ...,0.890414
2,aspiring human resources professional raleigh ...,0.886353
12,aspiring human resources management student se...,0.878086
24,aspiring human resources professional | passio...,0.876692
30,aspiring human resources professional | energe...,0.864373
20,business management major aspiring human resou...,0.863725
46,seek human resources position las vegas nevada...,0.863497
13,seek human resources opportunities chicago ill...,0.860379


In [24]:
score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    X_e5
)

df["similarity_2_5e"] = score_2.flatten()

df[df["similarity_2_5e"]>0.3][["clean_job_title", "similarity_2_5e"]].sort_values(
    by="similarity_2_5e",
    ascending=False
).head(10)

,clean_job_title,similarity_2_5e
13,seek human resources opportunities chicago ill...,0.892398
46,seek human resources position las vegas nevada...,0.890624
21,aspiring human resources manager seek internsh...,0.882211
8,seek human resources hris generalist positions...,0.876815
47,aspiring human resources manager | graduating ...,0.872307
42,seek human resources opportunity open travel...,0.872267
5,aspiring human resources specialist greater ne...,0.865797
12,aspiring human resources management student se...,0.865030
24,aspiring human resources professional | passio...,0.860166
23,nortia staffing seek human resources payroll a...,0.859875


In [25]:
tokenizer = AutoTokenizer.from_pretrained(
    "BAAI/bge-small-en-v1.5"
)

bge = AutoModel.from_pretrained(
    "BAAI/bge-small-en-v1.5"
)


def get_title_vector_bge(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = bge(**inputs)

    # Token embeddings
    token_embeddings = outputs.last_hidden_state

    # Attention mask
    attention_mask = inputs["attention_mask"]

    # Expand mask to same dimensions as embeddings
    mask = attention_mask.unsqueeze(-1).expand(
        token_embeddings.size()
    ).float()

    # Mean pooling
    vector = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

    return vector.squeeze().numpy() 


df["bge_for_modeling"] = df["clean_job_title"].apply(
    get_title_vector_bge
)


X_bge = np.vstack(df["bge_for_modeling"])

y1_vector = get_title_vector_bge(measure_rank_1)
y2_vector = get_title_vector_bge(measure_rank_2)

score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_bge
)

df["similarity_1_bge"] = score_1.flatten()
# Chain the filtering and sorting together
df[df["similarity_1_bge"] > 0.3][["clean_job_title", "similarity_1_bge"]].sort_values(
    by="similarity_1_bge",
    ascending=False
).head(10)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,clean_job_title,similarity_1_bge
2,aspiring human resources professional raleigh ...,0.786999
21,aspiring human resources manager seek internsh...,0.783562
24,aspiring human resources professional | passio...,0.780927
5,aspiring human resources specialist greater ne...,0.778894
12,aspiring human resources management student se...,0.767548
47,aspiring human resources manager | graduating ...,0.755488
13,seek human resources opportunities chicago ill...,0.754207
20,business management major aspiring human resou...,0.747347
30,aspiring human resources professional | energe...,0.743157
6,student humber college aspiring human resource...,0.731464


In [26]:
score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    X_bge
)

df["similarity_2_bge"] = score_2.flatten()

df[df["similarity_2_bge"] > 0.3][["clean_job_title", "similarity_2_bge"]].sort_values(
    by="similarity_2_bge",
    ascending=False
).head(10)

,clean_job_title,similarity_2_bge
13,seek human resources opportunities chicago ill...,0.828740
8,seek human resources hris generalist positions...,0.774525
21,aspiring human resources manager seek internsh...,0.768068
46,seek human resources position las vegas nevada...,0.761926
2,aspiring human resources professional raleigh ...,0.758372
42,seek human resources opportunity open travel...,0.736612
5,aspiring human resources specialist greater ne...,0.730112
24,aspiring human resources professional | passio...,0.729630
15,human resources staffing recruiting profession...,0.724346
23,nortia staffing seek human resources payroll a...,0.721670


In [28]:
df

,id,job_title,location,connection,relevance,clean_job_title,similarity_1_bow,similarity_2_bow,similarity_1_tfidf,similarity_2_tfidf,...,similarity_2_bert,sbert_for_modeling,similarity_1_sbert,similarity_2_sbert,e5_for_modeling,similarity_1_5e,similarity_2_5e,bge_for_modeling,similarity_1_bge,similarity_2_bge
0,1,2019 C.T. Bauer College of Business Graduate (...,"Houston, Texas",85,1,2019 c.t. bauer college business graduate magn...,0.298142,0.365148,0.094907,0.152204,...,0.557867,"[-0.3368231, 0.100788884, 0.01481131, 0.007473...",0.386246,0.284448,"[-0.5350845, 0.35418215, 0.55834466, 0.0453751...",0.822847,0.797523,"[-0.42292202, 0.31027558, 0.06566295, -0.41557...",0.611751,0.582164
1,2,Native English Teacher at EPIK (English Progra...,Kanada,500,0,native english teacher epik english program ko...,0.000000,0.000000,0.000000,0.000000,...,0.576568,"[-0.017585965, 0.064741224, 0.1884237, -0.1230...",0.139189,0.145103,"[-0.43027866, 0.36420888, 0.1779817, -0.023868...",0.760539,0.740750,"[-0.1163655, 0.31531736, 0.54091394, -0.209717...",0.488241,0.470397
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,1,aspiring human resources professional raleigh ...,0.547723,0.447214,0.365241,0.227746,...,0.621958,"[0.07544383, 0.13186923, -0.2456371, 0.6118936...",0.819170,0.734525,"[-0.59000343, 0.26603743, 0.15078649, 0.269511...",0.886353,0.854755,"[-0.44915116, 0.5947395, -0.099768125, -0.1236...",0.786999,0.758372
3,4,People Development Coordinator at Ryan,"Denton, Texas",500,0,people development coordinator ryan denton tex...,0.000000,0.000000,0.000000,0.000000,...,0.679889,"[-0.3321501, 0.055536296, -0.104886964, 0.0013...",0.323180,0.325776,"[-0.7640145, 0.54940265, 0.31106025, -0.079313...",0.789040,0.779482,"[-0.098992914, 0.042602587, -0.08287403, -0.31...",0.552727,0.562944
4,5,Advisory Board Member at Celal Bayar University,"Izmir, Turkiye",500,0,advisory board member celal bayar university i...,0.000000,0.000000,0.000000,0.000000,...,0.348195,"[-0.28228885, 0.22279498, -0.029336547, -0.049...",0.215421,0.204176,"[-0.43927935, 0.32124734, 0.19381846, 0.028976...",0.744379,0.743903,"[-0.048286103, 0.4516349, 0.3459198, -0.360809...",0.461143,0.446429
5,6,Aspiring Human Resources Specialist,Greater New York City Area,1,1,aspiring human resources specialist greater ne...,0.577350,0.471405,0.416505,0.259712,...,0.643626,"[0.0869284, -0.13570309, 0.04888661, 0.4970742...",0.788344,0.717399,"[-0.64446765, 0.43988165, 0.42056322, 0.136523...",0.900310,0.865797,"[0.22884524, 0.28346363, 0.5874645, -0.3783340...",0.778894,0.730112
6,7,Student at Humber College and Aspiring Human R...,Kanada,61,1,student humber college aspiring human resource...,0.577350,0.471405,0.345649,0.215529,...,0.641462,"[-0.03762171, -0.056122027, 0.065703966, 0.283...",0.702629,0.617134,"[-0.59977627, 0.080728956, 0.16360243, 0.18847...",0.850906,0.837079,"[-0.03341784, 0.36322445, 0.14161113, -0.51592...",0.731464,0.669440
7,8,HR Senior Specialist,San Francisco Bay Area,500,1,hr senior specialist san francisco bay area 500,0.000000,0.000000,0.000000,0.000000,...,0.610220,"[0.006473148, 0.48212543, 0.013325644, 0.14524...",0.411216,0.441064,"[-0.4605673, 0.47696906, 0.2891393, 0.15848958...",0.817140,0.806993,"[0.105418585, 0.26065356, -0.009833904, -0.401...",0.607978,0.642680
8,10,Seeking Human Resources HRIS and Generalist Po...,Greater Philadelphia Area,500,1,seek human resources hris generalist positions...,0.365148,0.447214,0.143339,0.229876,...,0.698048,"[-0.0032115537, -0.0290977, 0.012220966, 0.170...",0.629084,0.699338,"[-0.55382806, 0.23559651, 0.33015388, 0.148733...",0.846795,0.876815,"[0.022809764, 0.43152207, 0.3962153, -0.587740...",0.688832,0.774525
9,11,Student at Chapman University,"Lake Forest, California",2,0,student chapman university lake forest califor...,0.000000,0.000000,0.000000,0.000000,...,0.540106,"[0.38610727, -0.08744196, 0.11742731, 0.015188...",0.261305,0.196519,"[-0.54346424, 0.38001978, 0.43

In [29]:
df.to_csv("../data/vectors_hr_data.csv", index=False)